# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook loads and explores the [FAIR\^2 dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
We begin by loading the metadata and inspecting the dataset using `mlcroissant`. This will retrieve the Croissant metadata and provide basic information about the dataset and its structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Show core metadata (using attributes, not subscripting!)
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}\n")
print(f"License: {dataset.metadata.license}\n")
print(f"Version: {dataset.metadata.version}\n")
print(f"Cite As: {dataset.metadata.citeAs}\n")

## 2. Data Overview
Let's enumerate the available record sets and their fields within this dataset. Each entity's unique identifier is its `@id`. The mlcroissant Dataset metadata provides access to these elements using the schema structure.

In [ ]:
# List all RecordSets defined by @id
if hasattr(dataset.metadata, 'recordSet'):
    record_sets = dataset.metadata.recordSet
    print(f'Found {len(record_sets)} record set(s).\n')
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        if 'field' in rs:
            print('  Field @ids:')
            for f in rs['field']:
                fid = f['@id'] if isinstance(f, dict) and '@id' in f else f
                print(f'    - {fid}')
        else:
            print('  (No fields listed in schema)')
else:
    print('No record sets found! Please check the Croissant schema.')

## 3. Data Extraction
We can load records from each `RecordSet` identified above. We'll use the RecordSet(s) and field `@id`s. For demonstration purposes, we'll extract all records from the first available record set, using its `@id`.

In [ ]:
# Extract all records from the first available RecordSet
dataframes = {}
record_set_ids = []
if hasattr(dataset.metadata, 'recordSet'):
    for rs in dataset.metadata.recordSet:
        record_set_ids.append(rs['@id'])

    for recset_id in record_set_ids:
        print(f'Loading record set: {recset_id} ...')
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
else:
    print('No record sets found!')

# Print column names (fields as @id) for first record set
if record_set_ids:
    main_record_set = record_set_ids[0]
    print(f"\nColumns (field @ids) in {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())

    # Show first few rows
    dataframes[main_record_set].head()
else:
    print('No records to preview.')

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate EDA on a numeric field. Please identify a numeric field `@id` from the Data Overview (from above). We'll filter, normalize, and group data as examples. All column references use field `@id` exactly as returned.

*Update the variables below as needed to analyze relevant fields for your use-case.*

In [ ]:
# --- Edit these as needed for your dataset ---
# Use field @ids exactly as in the columns printed above
record_set_id = main_record_set  # use the 1st record set

# Example: use 'cr:coverage_percentage' and 'cr:accession' if available
numeric_field = None
group_field = None
for col in dataframes[record_set_id].columns:
    if (
        'coverage' in col or 'percent' in col or 'MW' in col or 'number' in col or 'count' in col
    ):
        numeric_field = col
    if (
        col == 'cr:accession' or 'accession' in col or 'group' in col
    ):
        group_field = col

if numeric_field is not None:
    print(f"Numeric field identified for analysis: {numeric_field}")
    threshold = dataframes[record_set_id][numeric_field].mean() if pd.api.types.is_numeric_dtype(dataframes[record_set_id][numeric_field]) else 10
    # Convert column to numeric if not already
    dataframes[record_set_id][numeric_field] = pd.to_numeric(dataframes[record_set_id][numeric_field], errors='coerce')
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group if group_field exists
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print('No obvious numeric field found. Please set `numeric_field` to a field @id such as a value or percentage column.')

## 5. Visualization
Let's plot the selected numeric field (and optionally, its grouped means) to visually inspect its distribution and any group-wise variation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None and group_field in dataframes[record_set_id].columns:
        plt.figure(figsize=(10,4))
        means = dataframes[record_set_id].groupby(group_field)[numeric_field].mean().sort_values()
        means.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print('Cannot plot because a numeric field was not found. Check and set `numeric_field` in the EDA section.')

## 6. Conclusion
We loaded and explored the FAIR\^2 dataset Mass Spectrometry Analysis using `mlcroissant`. Using only the schema's `@id` references, we examined the dataset structure, loaded tabular data, performed basic normalization and grouping, and visualized numeric attributes.

**Next steps:** You can extend this template by drilling into other record sets, testing more fields, or applying custom domain-specific analysis & ML workflows.